**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 09 â€” Evaluation & Ablation Study

Loads all trained checkpoints and produces:
- Full metric table (accuracy, F1, AUROC, precision, recall)
- ROC curves
- Confusion matrices
- Ablation comparison bar chart
- Optimal threshold search
- Failure case sampling

In [ ]:
import os
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score, recall_score,
    roc_curve, auc, confusion_matrix, classification_report,
)
from transformers import CLIPModel, CLIPProcessor

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    default_candidate = kaggle_input / "meta-hateful-meme-detection" / "data"
    if _looks_like_dataset_root(default_candidate):
        return default_candidate, "default:/kaggle/input/meta-hateful-meme-detection/data"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


def resolve_split(base_dir, names):
    base_dir = Path(base_dir)
    for name in names:
        path = base_dir / name
        if path.is_file():
            return path
    for name in names:
        matches = sorted(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
DEV_PATH = resolve_split(DATA_DIR, JSONL_CANDIDATES["dev"])
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CFG = {"clip_model": "openai/clip-vit-base-patch32", "max_text_len": 77, "batch_size": 64}

if DEV_PATH is None:
    raise FileNotFoundError(f"Expected dev split under {DATA_DIR}")

DATA_DIR = str(DATA_DIR)
IMG_DIR = str(IMG_DIR) if IMG_DIR is not None else None
DEV_PATH = str(DEV_PATH)
OUTPUT_DIR = str(OUTPUT_DIR)

print(f"Using DATA_DIR : {DATA_DIR}")
print(f"Using IMG_DIR  : {IMG_DIR}")
print(f"Using source   : {data_source}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Device         : {DEVICE}")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])


def clean_text(text):
    if not isinstance(text, str):
        return "[no text]"
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    return re.sub(r"\s+", " ", text).strip() or "[no text]"


dev_df = load_jsonl(DEV_PATH)
dev_df["clean_text"] = dev_df["text"].apply(clean_text)
print(f"Dev set: {len(dev_df)} samples")

In [ ]:
from pathlib import Path


def resolve_image_path(data_dir, image_ref):
    data_dir = Path(data_dir)
    image_ref = Path(str(image_ref))

    candidates = []
    if image_ref.is_absolute():
        candidates.append(image_ref)

    candidates.extend([
        data_dir / image_ref,
        data_dir.parent / image_ref,
    ])

    if image_ref.parts:
        if image_ref.parts[0] in {"img", "images"} and len(image_ref.parts) > 1:
            stripped = Path(*image_ref.parts[1:])
            candidates.extend([
                data_dir / stripped,
                data_dir.parent / stripped,
            ])
        elif image_ref.parts[0] not in {"img", "images"}:
            candidates.extend([
                data_dir / "img" / image_ref,
                data_dir / "images" / image_ref,
                data_dir.parent / "img" / image_ref,
                data_dir.parent / "images" / image_ref,
            ])

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find image '{image_ref}' relative to {data_dir}")

In [ ]:
#  Inline model definitions (same as notebook 08) 
class MemeDataset(Dataset):
    def __init__(self, df, data_dir, processor):
        self.df = df.reset_index(drop=True); self.data_dir = data_dir; self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try: img = Image.open(resolve_image_path(self.data_dir, row["img"])).convert("RGB")
        except: img = Image.new("RGB", (224, 224), 128)
        text = str(row.get("clean_text", row["text"]))
        enc  = self.processor(text=[text], images=img, return_tensors="pt",
                               padding="max_length", max_length=77, truncation=True)
        lbl  = int(row["label"]) if "label" in row else -1
        return {"pixel_values": enc["pixel_values"].squeeze(0),
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label": torch.tensor(lbl, dtype=torch.long)}

class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(CFG["clip_model"])
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self, pv, ids, mask):
        i = F.normalize(self.clip.get_image_features(pixel_values=pv), -1)
        t = F.normalize(self.clip.get_text_features(input_ids=ids, attention_mask=mask), -1)
        return i, t

class CrossAttentionFusion(nn.Module):
    def __init__(self, d=512, h=4):
        super().__init__()
        self.i2t = nn.MultiheadAttention(d, h, batch_first=True)
        self.t2i = nn.MultiheadAttention(d, h, batch_first=True)
        self.ni  = nn.LayerNorm(d); self.nt = nn.LayerNorm(d)
    def forward(self, i, t):
        is_ = i.unsqueeze(1); ts = t.unsqueeze(1)
        ic, ia = self.i2t(is_, ts, ts); tc, ta = self.t2i(ts, is_, is_)
        return torch.cat([self.ni(i+ic.squeeze(1)), self.nt(t+tc.squeeze(1))], -1), ia, ta

class ClassificationHead(nn.Module):
    def __init__(self, d=1024):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self, x): return self.net(x)

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CLIPEncoder(); self.fusion = CrossAttentionFusion(); self.head = ClassificationHead()
    def forward(self, pv, ids, mask):
        i,t = self.encoder(pv,ids,mask); f,ia,ta = self.fusion(i,t); return self.head(f),ia,ta

class ImageOnlyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(CFG["clip_model"])
        for p in self.clip.parameters(): p.requires_grad_(False)
        self.head = nn.Sequential(nn.Linear(512,256),nn.GELU(),nn.Dropout(0.3),nn.Linear(256,2))
    def forward(self, pv, ids, mask):
        return self.head(F.normalize(self.clip.get_image_features(pixel_values=pv),-1)), None, None

class TextOnlyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(CFG["clip_model"])
        for p in self.clip.parameters(): p.requires_grad_(False)
        self.head = nn.Sequential(nn.Linear(512,256),nn.GELU(),nn.Dropout(0.3),nn.Linear(256,2))
    def forward(self, pv, ids, mask):
        return self.head(F.normalize(self.clip.get_text_features(input_ids=ids,attention_mask=mask),-1)), None, None

class LateFusionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(CFG["clip_model"])
        for p in self.clip.parameters(): p.requires_grad_(False)
        self.head = nn.Sequential(nn.Linear(1024,512),nn.GELU(),nn.Dropout(0.3),
                                   nn.Linear(512,128),nn.GELU(),nn.Linear(128,2))
    def forward(self, pv, ids, mask):
        i=F.normalize(self.clip.get_image_features(pixel_values=pv),-1)
        t=F.normalize(self.clip.get_text_features(input_ids=ids,attention_mask=mask),-1)
        return self.head(torch.cat([i,t],-1)), None, None

print("Model classes loaded.")

In [ ]:
# â”€â”€ Load checkpoints and run inference â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
processor = CLIPProcessor.from_pretrained(CFG["clip_model"])
dev_ds    = MemeDataset(dev_df, DATA_DIR, processor)
dev_loader = DataLoader(dev_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=2)

VARIANTS = [
    ("cross_attention", HatefulMemeClassifier),
    ("image_only",      ImageOnlyClassifier),
    ("text_only",       TextOnlyClassifier),
    ("late_fusion",     LateFusionClassifier),
]

eval_results = {}

for name, ModelClass in VARIANTS:
    ckpt = os.path.join(OUTPUT_DIR, f"{name}_best.pt")
    if not os.path.exists(ckpt):
        print(f"SKIP {name} â€” checkpoint not found")
        continue
    model = ModelClass().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()

    all_probs, all_preds, all_labs = [], [], []
    with torch.no_grad():
        for batch in dev_loader:
            pv   = batch["pixel_values"].to(DEVICE)
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            logits, _, _ = model(pv, ids, mask)
            probs = torch.softmax(logits, -1)[:, 1].cpu().numpy()
            preds = logits.argmax(-1).cpu().numpy()
            all_probs.extend(probs); all_preds.extend(preds)
            all_labs.extend(batch["label"].numpy())

    eval_results[name] = {"probs": np.array(all_probs), "preds": np.array(all_preds),
                           "labels": np.array(all_labs)}
    del model; torch.cuda.empty_cache()
    print(f"Inference done: {name}")

In [ ]:
# â”€â”€ Metrics table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rows = []
for name, res in eval_results.items():
    labs, preds, probs = res["labels"], res["preds"], res["probs"]
    rows.append({
        "Model"      : name,
        "Accuracy"   : accuracy_score(labs, preds),
        "Precision"  : precision_score(labs, preds, pos_label=1),
        "Recall"     : recall_score(labs, preds, pos_label=1),
        "F1 (Hateful)": f1_score(labs, preds, pos_label=1),
        "Macro F1"   : f1_score(labs, preds, average="macro"),
        "AUROC"      : roc_auc_score(labs, probs),
    })

metrics_df = pd.DataFrame(rows)
print("\nABLATION STUDY RESULTS")
print("=" * 78)
print(metrics_df.to_string(index=False, float_format="{:.4f}".format))
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "ablation_results.csv"), index=False)

In [ ]:
# â”€â”€ ROC curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(8, 6))
colors  = {"cross_attention": "#F44336", "image_only": "#FF9800",
            "text_only": "#2196F3", "late_fusion": "#4CAF50"}

for name, res in eval_results.items():
    fpr, tpr, _ = roc_curve(res["labels"], res["probs"])
    auc_val     = auc(fpr, tpr)
    lw          = 2.5 if name == "cross_attention" else 1.5
    ax.plot(fpr, tpr, lw=lw, color=colors.get(name, "gray"),
            label=f"{name} (AUC={auc_val:.3f})")

ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves â€” All Variants", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# â”€â”€ Confusion matrices â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
n_variants = len(eval_results)
fig, axes = plt.subplots(1, n_variants, figsize=(5 * n_variants, 4))
if n_variants == 1: axes = [axes]

for ax, (name, res) in zip(axes, eval_results.items()):
    cm = confusion_matrix(res["labels"], res["preds"])
    sns.heatmap(cm, annot=True, fmt="d", ax=ax,
                xticklabels=["Non-hate", "Hate"],
                yticklabels=["Non-hate", "Hate"],
                cmap="Blues", cbar=False)
    ax.set_title(name, fontsize=10, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# â”€â”€ Optimal threshold search (for best model) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best_model_name = max(eval_results.keys(),
    key=lambda n: roc_auc_score(eval_results[n]["labels"], eval_results[n]["probs"]))

res     = eval_results[best_model_name]
labs    = res["labels"]
probs   = res["probs"]

thresholds = np.linspace(0.1, 0.9, 81)
f1_scores  = [f1_score(labs, (probs >= t).astype(int), pos_label=1) for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, color="#E91E63", lw=2)
plt.axvline(best_thresh, color="black", linestyle="--", label=f"Best threshold={best_thresh:.2f}")
plt.axvline(0.5, color="gray", linestyle=":", alpha=0.7, label="Default=0.5")
plt.xlabel("Decision threshold"); plt.ylabel("F1 (Hateful)")
plt.title(f"Threshold Tuning â€” {best_model_name}", fontweight="bold")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Default threshold (0.5) F1: {f1_score(labs, (probs>=0.5).astype(int), pos_label=1):.4f}")
print(f"Optimal threshold ({best_thresh:.2f}) F1: {max(f1_scores):.4f}")

In [ ]:
# â”€â”€ Ablation bar chart â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in [
    (axes[0], "AUROC",       "AUROC"),
    (axes[1], "F1 (Hateful)","F1 (Hateful)"),
    (axes[2], "Macro F1",    "Macro F1"),
]:
    bars = ax.bar(metrics_df["Model"], metrics_df[col],
                  color=[colors.get(m, "gray") for m in metrics_df["Model"]])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", fontsize=9)
    ax.set_ylim(0.4, 1.0)
    ax.set_title(title, fontweight="bold")
    ax.set_xticklabels(metrics_df["Model"], rotation=20, ha="right", fontsize=8)

plt.suptitle("Ablation Study â€” Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ablation_chart.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Evaluation notebook complete. Proceed to notebook 10 (Explainability).")